# Simulação Financeira de Manutenção Preditiva

A simulação abaixo avalia o impacto econômico da substituição de uma estratégia de manutenção reativa (corretiva) por uma estratégia de manutenção preditiva baseada em um classificador XGBoost. A avaliação financeira é baseada em um conjunto de teste separado contendo 136 dias operacionais distintos. Durante esse período, o modelo produziu os seguintes resultados: 368 verdadeiros negativos, 0 falsos positivos, 1 falso negativo e 381 verdadeiros positivos, o que corresponde a um F1-score de 99%, indicando um equilíbrio quase perfeito entre precisão e recall. O modelo deixou de detectar apenas uma falha e não gerou falsos alarmes, o que influencia fortemente o resultado financeiro da simulação.

Para estimar o impacto mensal, os eventos observados no conjunto de teste foram normalizados para um período de 30 dias utilizando um fator de escala de 30/136. A partir dessa normalização, as falhas mensais no cenário reativo foram calculadas como (TP + FN), enquanto as intervenções preditivas mensais foram calculadas como (TP + FP), e as falhas não detectadas mensais como FN. Essa abordagem assume que a taxa de falhas observada durante o período de teste é representativa de um mês típico de operação, o que constitui uma premissa importante do modelo. Além disso, assume-se que a predição ocorre em tempo hábil para agir em manutenção preditiva. 

A simulação deixa livre para experimentar, os custos de impacto por hora parada, custos do sistema de machine learning, custos de operação durante manutenções (ambas reativa e preditiva), além das durações de manutenções nos dois cenários (ambas reativa e preditiva).

## Abordagem dos Cálculos

Os custos de eventos foram calculados multiplicando a duração do evento (em horas) pela soma da perda de receita de produção por hora e do custo operacional de manutenção por hora.

- Na estratégia reativa, cada falha resulta em downtime não planejado, de modo que o custo mensal total é igual ao número de falhas mensais multiplicado pelo custo de downtime por evento.

- Na estratégia preditiva, os custos incluem manutenção preventiva para cada falha prevista (verdadeiros positivos e falsos positivos), custos de downtime para falhas não detectadas (falsos negativos) e o custo operacional mensal do sistema de machine learning.

O benefício financeiro líquido é calculado como a diferença entre o custo do cenário reativo e o custo total do cenário preditivo, enquanto o ROI é definido como o lucro líquido dividido pelo custo mensal do modelo.

Dessa forma, ambas as abordagens são avaliadas sob o mesmo volume observado de falhas, os eventos são adequadamente normalizados para um horizonte temporal comum e os custos são calculados utilizando as mesmas premissas estruturais. 

Utilize o widget abaixo para experimentar diferentes premissas e avaliar cenários financeiros alternativos.

In [1]:
# Imports and functions
import json
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, Markdown

# Load Data
test_set_csv = "https://raw.githubusercontent.com/monicaneli/Manutencao-Preditiva-em-Maquinas-Industriais/refs/heads/main/simulador/X_test_with_dates.csv"
X_test_with_dates = pd.read_csv(test_set_csv)

confusion_matrix_file = "https://raw.githubusercontent.com/monicaneli/Manutencao-Preditiva-em-Maquinas-Industriais/main/simulador/confusion_matrix.json"
data = pd.read_json(confusion_matrix_file)
conf_matrix = data["confusion_matrix"].tolist()

# Proper format
conf_matrix = [[368,0],[1,381]]

distinct_days_test = X_test_with_dates['Day'].nunique()

# Financial Functions
def calculate_event_cost(hours, production_cost_per_hour, operational_cost_per_hour):
    return hours * (production_cost_per_hour + operational_cost_per_hour)


def simulate_financial_impact(conf_matrix,
                              distinct_days_test,
                              production_cost_per_hour,
                              cost_model,
                              downtime_hours,
                              predictive_hours,
                              downtime_operational_cost,
                              predictive_operational_cost):

    TN, FP = conf_matrix[0]
    FN, TP = conf_matrix[1]

    scaling_factor = 30 / distinct_days_test

    monthly_failures = (TP + FN) * scaling_factor
    monthly_predictive_actions = (TP + FP) * scaling_factor
    monthly_false_negatives = FN * scaling_factor

    downtime_cost = calculate_event_cost(
        downtime_hours,
        production_cost_per_hour,
        downtime_operational_cost
    )

    predictive_cost = calculate_event_cost(
        predictive_hours,
        production_cost_per_hour,
        predictive_operational_cost
    )

    baseline_cost = monthly_failures * downtime_cost

    predictive_total_cost = (
        monthly_predictive_actions * predictive_cost +
        monthly_false_negatives * downtime_cost +
        cost_model
    )

    net_profit = baseline_cost - predictive_total_cost
    roi = net_profit / cost_model if cost_model > 0 else 0

    return {
        "TN": TN,
        "FP": FP,
        "FN": FN,
        "TP": TP,
        "monthly_failures": monthly_failures,
        "monthly_predictive_actions": monthly_predictive_actions,
        "monthly_false_negatives": monthly_false_negatives,
        "downtime_cost": downtime_cost,
        "predictive_cost": predictive_cost,
        "baseline_cost": baseline_cost,
        "predictive_total_cost": predictive_total_cost,
        "net_profit": net_profit,
        "roi": roi,
        "is_viable": net_profit > 0
    }


# Result Display
def display_financial_results(results):

    viability = "✅ Economicamente Viável" if results["is_viable"] else "❌ Não Economicamente Viável"

    display(Markdown("### Manutenção Preditiva vs Manutenção Reativa"))

    display(Markdown(f"""
**Matriz de Confusão**

TN: {results['TN']} | FP: {results['FP']}  
FN: {results['FN']} | TP: {results['TP']}

**Conjunto de Teste:** {distinct_days_test} dias distintos
"""))

    display(Markdown("### Eventos Mensais Normalizados"))

    display(pd.DataFrame({
        "Métrica":[
            "Falhas (Cenário Reativo)",
            "Intervenções Preditivas",
            "Falhas Não Detectadas"
        ],
        "Valor":[
            round(results["monthly_failures"],2),
            round(results["monthly_predictive_actions"],2),
            round(results["monthly_false_negatives"],2)
        ]
    }))

    display(Markdown("### Comparação Financeira"))

    display(pd.DataFrame({
        "Métrica":[
            "Custo Mensal (Manutenção Reativa)",
            "Custo Mensal (Manutenção Preditiva)",
            "Benefício Financeiro Mensal",
            "ROI"
        ],
        "Valor":[
            f"${results['baseline_cost']:,.0f}",
            f"${results['predictive_total_cost']:,.0f}",
            f"${results['net_profit']:,.0f}",
            f"{results['roi']*100:.2f}%"
        ]
    }))

    display(Markdown(f"### {viability}"))


In [2]:
# Unit tests
def test_net_profit_positive():
    
    results = simulate_financial_impact(
        conf_matrix,
        distinct_days_test,
        production_cost_per_hour=5000,
        cost_model=5000,
        downtime_hours=5,
        predictive_hours=4,
        downtime_operational_cost=950,
        predictive_operational_cost=750
    )
    
    assert results["net_profit"] > 0
    assert results["is_viable"]  == True

def test_net_profit_negative2():
    
    results = simulate_financial_impact(
        conf_matrix,
        distinct_days_test,
        production_cost_per_hour=1000,
        cost_model=1000,
        downtime_hours=1,
        predictive_hours=1,
        downtime_operational_cost=50,
        predictive_operational_cost=50
    )
    
    assert results["net_profit"] < 0
    assert results["is_viable"]  == False



def test_net_profit_negative():
    
    # Force model to be expensive and low impact
    results = simulate_financial_impact(
        conf_matrix,
        distinct_days_test,
        production_cost_per_hour=500,
        cost_model=500000,  # extremely high model cost
        downtime_hours=1,
        predictive_hours=1,
        downtime_operational_cost=50,
        predictive_operational_cost=50
    )
    
    assert results["net_profit"] < 0
    assert results["is_viable"] == False

test_net_profit_positive()
test_net_profit_negative()
test_net_profit_negative2()

In [3]:
# Widgets
style = {'description_width': '300px'}
layout = widgets.Layout(width='600px')

# Sliders
production_cost_slider = widgets.IntSlider(
    min=1000, max=20000, step=100, value=1000,
    description='Impacto na Receita por Hora',
    style=style, layout=layout
)

model_cost_slider = widgets.IntSlider(
    min=1000, max=20000, step=1000, value=1000,
    description='Custo Mensal do Sistema de ML',
    style=style, layout=layout
)

downtime_hours_slider = widgets.FloatSlider(
    min=1, max=12, step=0.5, value=1,
    description='Duração do Downtime (horas)',
    style=style, layout=layout
)

predictive_hours_slider = widgets.FloatSlider(
    min=1, max=12, step=0.5, value=1,
    description='Duração da Manutenção Preditiva',
    style=style, layout=layout
)

downtime_cost_slider = widgets.IntSlider(
    min=50, max=5000, step=50, value=100,
    description='Custo Operacional do Downtime / hora',
    style=style, layout=layout
)

predictive_cost_slider = widgets.IntSlider(
    min=50, max=5000, step=50, value=50,
    description='Custo Operacional da Manutenção Preditiva / hora',
    style=style, layout=layout
)


# Interactive Simulation
output = widgets.Output()

def run_simulation(change=None):

    with output:
        output.clear_output()

        results = simulate_financial_impact(
            conf_matrix,
            distinct_days_test,
            production_cost_slider.value,
            model_cost_slider.value,
            downtime_hours_slider.value,
            predictive_hours_slider.value,
            downtime_cost_slider.value,
            predictive_cost_slider.value
        )

        display_financial_results(results)


for widget in [
    production_cost_slider,
    model_cost_slider,
    downtime_hours_slider,
    predictive_hours_slider,
    downtime_cost_slider,
    predictive_cost_slider
]:
    widget.observe(run_simulation, "value")


# Dashboard Layout
controls = widgets.VBox([
    production_cost_slider,
    model_cost_slider,
    downtime_hours_slider,
    predictive_hours_slider,
    downtime_cost_slider,
    predictive_cost_slider
])

display(Markdown("## Simulador Interativo"))

display(Markdown("""
Ajuste os parâmetros abaixo para comparar os custos entre **manutenção reativa e manutenção preditiva**.
"""))

display(controls)

display(output)

run_simulation()

## Simulador Interativo


Ajuste os parâmetros abaixo para comparar os custos entre **manutenção reativa e manutenção preditiva**.


Output()

## Principais resultados

Dentro dessa abordagem, a estratégia preditiva mostra-se economicamente viável na maioria dos cenários de custo testados. No entanto, o elevado ROI observado na simulação deve ser interpretado com cautela.

O ROI reportado parece elevado principalmente por três fatores:

- o desempenho preditivo quase perfeito (F1-score de 99%, com praticamente nenhuma falha não detectada ou falso alarme);

- a frequência relativamente alta de eventos de falha observada no conjunto de dados (o que amplifica a economia acumulada quando escalada para um mês);

- o custo operacional relativamente baixo assumido para o modelo. Mesmo pequenas diferenças de custo por evento acumulam-se significativamente quando aproximadamente 84 falhas mensais são consideradas.

Outras conclusões podem ser extraídas além da comparação financeira:

- a atratividade econômica da estratégia preditiva é altamente sensível ao desempenho do modelo;

- aumentos na quantidade de falsos negativos elevariam rapidamente os custos de downtime, enquanto aumentos de falsos positivos inflariam os custos de manutenção desnecessária;

- os resultados dependem fortemente da premissa de que a frequência de falhas observada no conjunto de teste reflete a realidade operacional de longo prazo;

- o custo do modelo considerado na simulação inclui apenas despesas operacionais mensais e exclui custos de desenvolvimento, infraestrutura, integração, retreinamento e mudanças organizacionais. A inclusão desses elementos provavelmente reduziria o ROI no curto prazo;

- embora o modelo foque no impacto financeiro direto, a manutenção preditiva pode gerar benefícios indiretos, como maior estabilidade de produção, redução da volatilidade operacional, melhor capacidade de planejamento e menor pressão sobre as equipes de manutenção — efeitos que não estão capturados na formulação atual de custos.

Em conclusão, com base na lógica financeira da simulação, a manutenção preditiva mostra-se economicamente atrativa sob as premissas estabelecidas.

Entretanto, a magnitude do ROI deve ser interpretada como um cenário otimista, impulsionado por um desempenho preditivo extremamente forte e por premissas de custo simplificadas. Uma implementação industrial completa exigiria um modelo de custos mais abrangente e análises de sensibilidade para produzir uma avaliação de investimento mais conservadora e realista.

# Conclusão

O conjunto de dados forneceu uma boa base para a previsão de downtime, permitindo a identificação de padrões relevantes e das variáveis mais influentes associadas às falhas das máquinas.

O modelo XGBoost alcançou alta acurácia e um F1-score elevado, ambos próximos de 99,99%, conseguindo prever efetivamente o downtime das máquinas com base em dados de sensores e permitindo que operadores tomem ações preventivas antes que as falhas ocorram.

A simulação, considerando fatores financeiros e viabilidade econômica, indica que, com base no conjunto de teste de 136 dias e em um modelo XGBoost com F1-score de 99%, a manutenção preditiva é financeiramente superior à manutenção reativa sob as premissas testadas. Após a normalização mensal, a redução do downtime não planejado gera benefício financeiro líquido positivo na maioria dos cenários de custo. No entanto, o alto ROI deve ser interpretado com cautela, pois reflete desempenho preditivo quase perfeito, frequência relativamente elevada de falhas observadas e premissas de custo simplificadas.

Para aumentar o realismo da simulação, versões futuras devem incorporar:

- uma premissa de lead time acionável (tempo mínimo necessário para agir antes do downtime);
- um threshold operacional de probabilidade para disparar intervenções de manutenção;
- redução parcial de severidade (uma vez que nem todas as falhas podem ser totalmente evitadas);
- restrições operacionais, como limites de intervenções.

Além disso, incluir uma política de manutenção programada (por exemplo, manutenção a cada três dias) permitiria uma comparação completa entre estratégias corretivas, preditivas e preventivas. A ampliação da estrutura de custos para incluir desenvolvimento, integração e despesas operacionais de longo prazo também proporcionaria uma avaliação financeira mais conservadora e adequada para tomada de decisão executiva.

Em conclusão, com base na lógica financeira da simulação, a manutenção preditiva mostra-se economicamente atrativa sob as premissas iniciais consideradas.

_Obrigado pela leitura!_